# A Data-Driven Analysis of Tourist Satisfaction in Sri Lankan Destinations Using Online Review Analytics

## 1. Import Libraries

In [1]:
# Core Libraries
import pandas as pd
import numpy as np
import os
import re
import json
import random
import string
import subprocess
import sys
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, Optional, Tuple
from urllib import error, request

# Data Processing & NLP
import pycountry
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.sentiment import SentimentIntensityAnalyzer

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.feature_extraction.text import CountVectorizer
from gensim import corpora
from gensim.models import LdaModel
from wordcloud import WordCloud

# Transformers & Deep Learning
from transformers import pipeline
import torch
import warnings
warnings.filterwarnings('ignore')

# Display Settings
pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")

# NLTK Downloads
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('vader_lexicon', quiet=True)

# Reproducibility
random.seed(42)
np.random.seed(42)

print("✓ All libraries imported successfully")

c:\Users\abaiy\Group-J---Research-Paper\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ All libraries imported successfully


## 2. Load & Explore Data

In [2]:
# Load raw data
df = pd.read_csv("Reviews.csv", encoding='latin1')

# Basic inspection
print("📊 Dataset Overview:")
print(f"  Shape: {df.shape}")
print(f"  Columns: {df.columns.tolist()}")
print(f"\n📋 Data Types:")
print(df.dtypes)
print(f"\n❌ Missing Values:")
print(df.isnull().sum())
print(f"\n⚠️  Duplicates: {df.duplicated().sum()}")
print(f"\n⭐ Rating Distribution:")
print(df["Rating"].value_counts().sort_index())
print(f"\n📍 Top 10 Cities:")
print(df["Located_City"].value_counts().head(10))

📊 Dataset Overview:
  Shape: (16156, 14)
  Columns: ['Location_Name', 'Located_City', 'Location', 'Location_Type', 'User_ID', 'User_Location', 'User_Locale', 'User_Contributions', 'Travel_Date', 'Published_Date', 'Rating', 'Helpful_Votes', 'Title', 'Text']

📋 Data Types:
Location_Name           str
Located_City            str
Location                str
Location_Type           str
User_ID                 str
User_Location           str
User_Locale             str
User_Contributions    int64
Travel_Date             str
Published_Date          str
Rating                int64
Helpful_Votes         int64
Title                   str
Text                    str
dtype: object

❌ Missing Values:
Location_Name         0
Located_City          0
Location              0
Location_Type         0
User_ID               0
User_Location         0
User_Locale           0
User_Contributions    0
Travel_Date           0
Published_Date        0
Rating                0
Helpful_Votes         0
Title          

## 3. Feature Engineering: Location Extraction

In [3]:
# Define location mapping databases
provinces = [
    "Western Province", "Central Province", "Southern Province",
    "Northern Province", "Eastern Province", "North Western Province",
    "North Central Province", "Uva Province", "Sabaragamuwa Province",
]

city_to_district = {
    "Arugam Bay": "Ampara", "Colombo": "Colombo", "Kandy": "Kandy",
    "Nuwara Eliya": "Nuwara Eliya", "Galle": "Galle", "Mirissa": "Matara",
    "Ella": "Badulla", "Negombo": "Gampaha", "Polonnaruwa": "Polonnaruwa",
    "Sigiriya": "Matale", "Trincomalee": "Trincomalee", "Jaffna": "Jaffna",
    "Batticaloa": "Batticaloa", "Anuradhapura": "Anuradhapura", "Matara": "Matara",
    "Kalutara": "Kalutara", "Bentota": "Galle", "Hikkaduwa": "Galle",
    "Mirigama": "Gampaha", "Habarana": "Anuradhapura",
}

manual_location_mappings = {
    "Udawalawe National Park": {"province": "Sabaragamuwa Province", "district": "Ratnapura"},
    "North Central Province": {"province": "North Central Province", "district": "Anuradhapura"},
}

province_pattern = re.compile(
    r"(" + "|".join([re.escape(p) for p in provinces]) + r")", flags=re.IGNORECASE
)

def extract_province(location_str):
    """Extract province from location string"""
    if not isinstance(location_str, str) or not location_str.strip():
        return None
    
    loc = location_str.strip()
    
    # Manual mapping
    for k, v in manual_location_mappings.items():
        if k.lower() == loc.lower():
            return v.get("province")
    
    # Pattern matching
    m = province_pattern.search(loc)
    if m:
        return m.group(1)
    
    # Numeric pattern
    m2 = re.search(r"([A-Za-z ]+Province)\b", loc)
    if m2:
        return m2.group(1).strip()
    
    return None

def infer_district(row):
    """Infer district from location and city"""
    city = row.get("Located_City")
    location = row.get("Location")
    
    # Manual mapping
    if isinstance(location, str):
        for k, v in manual_location_mappings.items():
            if k.lower() == location.strip().lower():
                return v.get("district")
    
    # City mapping
    if isinstance(city, str) and city in city_to_district:
        return city_to_district[city]
    
    if not isinstance(location, str):
        return None
    
    parts = [p.strip() for p in location.split(",") if p.strip()]
    
    # Check for explicit district
    for part in parts:
        if "District" in part:
            return part.replace("District", "").strip()
    
    # Infer from position
    if len(parts) >= 3:
        return parts[-2]
    elif len(parts) == 2:
        return parts[0]
    elif len(parts) == 1 and not any(parts[0].lower() == p.lower() for p in provinces):
        return parts[0]
    
    return None

# Apply location extraction
print("🔄 Extracting province and district...")
df["province"] = df["Location"].apply(extract_province)
df["district"] = df.apply(infer_district, axis=1)

print(f"✓ Province coverage: {df['province'].notna().sum()}/{len(df)} rows")
print(f"✓ District coverage: {df['district'].notna().sum()}/{len(df)} rows")
print(f"\n📍 Top Provinces:")
print(df["province"].value_counts().head(10))

🔄 Extracting province and district...
✓ Province coverage: 16156/16156 rows
✓ District coverage: 16156/16156 rows

📍 Top Provinces:
province
Central Province          5252
North Central Province    3171
Southern Province         2648
Western Province          1890
Eastern Province          1162
Uva Province              1040
Sabaragamuwa Province      518
Northern Province          475
Name: count, dtype: int64


## 4. User Location Parsing: Country & Region

In [4]:
# Install pycountry if needed
try:
    import pycountry
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pycountry"])
    import pycountry

@dataclass
class ParseResult:
    country: Optional[str]
    other: Optional[str]
    method: str
    confidence: float

# Regional data
US_STATES = {"alabama", "alaska", "arizona", "arkansas", "california", "colorado",
             "connecticut", "delaware", "florida", "georgia", "hawaii", "idaho",
             "illinois", "indiana", "iowa", "kansas", "kentucky", "louisiana",
             "maine", "maryland", "massachusetts", "michigan", "minnesota",
             "mississippi", "missouri", "montana", "nebraska", "nevada",
             "new hampshire", "new jersey", "new mexico", "new york",
             "north carolina", "north dakota", "ohio", "oklahoma", "oregon",
             "pennsylvania", "rhode island", "south carolina", "south dakota",
             "tennessee", "texas", "utah", "vermont", "virginia", "washington",
             "west virginia", "wisconsin", "wyoming", "district of columbia", "puerto rico"}

AU_STATES = {"new south wales", "queensland", "south australia", "tasmania",
             "victoria", "western australia", "australian capital territory", "northern territory"}

CA_PROVINCES = {"alberta", "british columbia", "manitoba", "new brunswick",
                "newfoundland and labrador", "nova scotia", "ontario",
                "prince edward island", "quebec", "saskatchewan",
                "northwest territories", "nunavut", "yukon"}

INDIA_STATES = {"andhra pradesh", "arunachal pradesh", "assam", "bihar",
                "chhattisgarh", "goa", "gujarat", "haryana", "himachal pradesh",
                "jharkhand", "karnataka", "kerala", "madhya pradesh",
                "maharashtra", "manipur", "meghalaya", "mizoram", "nagaland",
                "odisha", "punjab", "rajasthan", "sikkim", "tamil nadu",
                "telangana", "tripura", "uttar pradesh", "uttarakhand",
                "west bengal", "delhi", "jammu and kashmir", "ladakh"}

UK_REGIONS = {"england", "scotland", "wales", "northern ireland"}

REGION_TO_COUNTRY = {
    **{state: "United States" for state in US_STATES},
    **{state: "Australia" for state in AU_STATES},
    **{state: "Canada" for state in CA_PROVINCES},
    **{state: "India" for state in INDIA_STATES},
    **{region: "United Kingdom" for region in UK_REGIONS},
    "new england": "United States",
}

# Build country index
def build_country_index():
    alias_map = {}
    for country in pycountry.countries:
        alias_map[country.name.lower()] = country.name
    
    manual = {
        "usa": "United States", "u.s.a": "United States", "us": "United States",
        "u.s": "United States", "uk": "United Kingdom", "u.k": "United Kingdom",
        "uae": "United Arab Emirates", "u.a.e": "United Arab Emirates",
        "russia": "Russian Federation", "viet nam": "Vietnam",
    }
    alias_map.update(manual)
    
    preferred = {
        "Korea, Republic of": "South Korea",
        "Korea, Democratic People's Republic of": "North Korea",
        "Russian Federation": "Russia", "Viet Nam": "Vietnam",
    }
    return alias_map, preferred

COUNTRY_ALIAS_TO_NAME, COUNTRY_SHORT_TO_PREFERRED = build_country_index()

def parse_user_location(raw_value):
    """Parse user location to extract country and region"""
    if raw_value is None or (isinstance(raw_value, float) and pd.isna(raw_value)):
        return ParseResult(country=None, other=None, method="empty", confidence=0.0)
    
    text = str(raw_value).strip()
    if not text:
        return ParseResult(country=None, other=None, method="empty", confidence=0.0)
    
    parts = [p.strip() for p in text.split(",") if p.strip()]
    country = None
    
    # Search for country
    for part in reversed(parts):
        if part.lower() in COUNTRY_ALIAS_TO_NAME:
            country = COUNTRY_SHORT_TO_PREFERRED.get(
                COUNTRY_ALIAS_TO_NAME[part.lower()],
                COUNTRY_ALIAS_TO_NAME[part.lower()]
            )
            break
    
    # Search for region
    region = None
    for part in reversed(parts):
        if part.lower() in REGION_TO_COUNTRY:
            region = part.title()
            if not country:
                country = REGION_TO_COUNTRY[part.lower()]
            break
    
    if country:
        return ParseResult(
            country=country, other=region,
            method="rule_based", confidence=0.9 if region else 0.85
        )
    
    return ParseResult(country=None, other=None, method="unresolved", confidence=0.0)

# Apply location parsing
print("🔄 Parsing user locations...")
parsed = df["User_Location"].apply(parse_user_location)
df["user_country"] = parsed.apply(lambda x: x.country)
df["user_region"] = parsed.apply(lambda x: x.other)
df["location_parse_confidence"] = parsed.apply(lambda x: x.confidence)

print(f"✓ Country resolution: {df['user_country'].notna().sum()}/{len(df)} rows")
print(f"\n🌍 Top 15 Countries:")
print(df["user_country"].value_counts().head(15))

🔄 Parsing user locations...
✓ Country resolution: 14494/16156 rows

🌍 Top 15 Countries:
user_country
United Kingdom          4530
Australia               2000
India                   1625
United States            950
United Arab Emirates     432
Canada                   396
Singapore                340
Germany                  307
New Zealand              240
France                   233
China                    214
Malaysia                 202
Ireland                  142
Belgium                  142
Switzerland              140
Name: count, dtype: int64


## 5. Feature Engineering: Temporal Features

In [5]:
# Convert dates
for col in ['Travel_Date', 'Published_Date']:
    df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
    df[f'{col}_Month'] = df[col].dt.month
    df[f'{col}_Year'] = df[col].dt.year

# Add text features
df['review_length'] = df['Text'].apply(len)
df['word_count'] = df['Text'].apply(lambda x: len(str(x).split()))
df['title_length'] = df['Title'].apply(len)

print("✓ Temporal features extracted")
print(f"\n📅 Date Range:")
print(f"  Travel: {df['Travel_Date'].min()} to {df['Travel_Date'].max()}")
print(f"  Published: {df['Published_Date'].min()} to {df['Published_Date'].max()}")

✓ Temporal features extracted

📅 Date Range:
  Travel: 2010-09-01 00:00:00+00:00 to 2023-05-01 00:00:00+00:00
  Published: 2011-03-12 12:00:09+00:00 to 2023-05-20 05:15:38+00:00


## 6. Data Cleanup & Consolidation

In [6]:
# Drop raw location columns (redundant with extracted features)
df = df.drop(columns=['Location', 'User_Location'], errors='ignore')

print(f"✓ Cleaned up redundant columns")
print(f"\n📊 Dataset Shape: {df.shape}")
print(f"\n📋 Current Columns ({len(df.columns)}):")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

✓ Cleaned up redundant columns

📊 Dataset Shape: (16156, 24)

📋 Current Columns (24):
   1. Location_Name
   2. Located_City
   3. Location_Type
   4. User_ID
   5. User_Locale
   6. User_Contributions
   7. Travel_Date
   8. Published_Date
   9. Rating
  10. Helpful_Votes
  11. Title
  12. Text
  13. province
  14. district
  15. user_country
  16. user_region
  17. location_parse_confidence
  18. Travel_Date_Month
  19. Travel_Date_Year
  20. Published_Date_Month
  21. Published_Date_Year
  22. review_length
  23. word_count
  24. title_length


## 7. Install & Load Sentiment Analysis Models

In [7]:
# Install required packages
required_packages = ["torch", "transformers>=4.30.0", "sentence-transformers>=2.2.2"]
print("Installing required packages...")

for package in required_packages:
    pkg_name = package.split('>')[0].split('<')[0].split('=')[0].strip()
    try:
        __import__(pkg_name.replace('-', '_'))
        print(f"  ✓ {pkg_name} already installed")
    except ImportError:
        print(f"  Installing {pkg_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        print(f"  ✓ {pkg_name} installed")

print("\n✓ All packages ready")

Installing required packages...
  ✓ torch already installed
  ✓ transformers already installed
  ✓ sentence-transformers already installed

✓ All packages ready


## 6b. Configuration: HuggingFace Token & Performance Settings

In [8]:
# HuggingFace Token Setup (optional but recommended)
import os
from getpass import getpass

hf_token = os.getenv('HF_TOKEN')
if not hf_token:
    print("⚠️  HF_TOKEN not set. HuggingFace operations will have lower rate limits.")
    print("   To set token: export HF_TOKEN='your_token_here' (Linux/Mac)")
    print("   Or: set HF_TOKEN=your_token_here (Windows)")
    print("   Get token at: https://huggingface.co/settings/tokens")
else:
    print("✓ HuggingFace token configured")

# Performance Settings
BATCH_SIZE = 32  # Increase for faster GPU, decrease for low memory
SAMPLE_SIZE = None  # Set to integer (e.g., 10000) to sample data, None for full dataset
ENABLE_PROGRESS_BARS = True  # Set False to hide tqdm progress
CACHE_MODELS = True  # Reuse loaded models

# Device configuration
device_name = "CUDA (GPU)" if torch.cuda.is_available() else "CPU"
print(f"\n🖥️  Computing Device: {device_name}")
if torch.cuda.is_available():
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Install tqdm for progress bars
try:
    from tqdm import tqdm
except ImportError:
    print("Installing tqdm for progress bars...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tqdm"])
    from tqdm import tqdm

print(f"\n✓ Performance settings configured")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Sample size: {SAMPLE_SIZE if SAMPLE_SIZE else 'Full dataset'}")
print(f"  Progress bars: {ENABLE_PROGRESS_BARS}")


⚠️  HF_TOKEN not set. HuggingFace operations will have lower rate limits.
   To set token: export HF_TOKEN='your_token_here' (Linux/Mac)
   Or: set HF_TOKEN=your_token_here (Windows)
   Get token at: https://huggingface.co/settings/tokens

🖥️  Computing Device: CPU

✓ Performance settings configured
  Batch size: 32
  Sample size: Full dataset
  Progress bars: True


In [9]:
# Load sentiment analysis models
print("Loading sentiment analysis models...")
print(f"Device: {'CUDA (GPU)' if torch.cuda.is_available() else 'CPU'}\n")

cardiff_sentiment = pipeline(
    "text-classification",
    model="cardiffnlp/twitter-roberta-base-sentiment",
    device=0 if torch.cuda.is_available() else -1
)
print("  ✓ Cardiff RoBERTa")

siebert_sentiment = pipeline(
    "text-classification",
    model="siebert/sentiment-roberta-large-english",
    device=0 if torch.cuda.is_available() else -1
)
print("  ✓ SieBERT")

bertweet_sentiment = pipeline(
    "text-classification",
    model="finiteautomata/bertweet-base-sentiment-analysis",
    device=0 if torch.cuda.is_available() else -1
)
print("  ✓ BERTweet")

try:
    absa_sentiment = pipeline(
        "text-classification",
        model="yangheng/deberta-v3-large-absa-v1.1",
        device=0 if torch.cuda.is_available() else -1
    )
    print("  ✓ DeBERTa ABSA")
except Exception as e:
    print(f"  ⚠️  DeBERTa ABSA failed, using Cardiff as fallback")
    absa_sentiment = cardiff_sentiment

emotion_classifier = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    device=0 if torch.cuda.is_available() else -1
)
print("  ✓ Emotion Classifier")

print("\n✓ All models loaded successfully!")

Loading sentiment analysis models...
Device: CPU



Loading weights: 100%|██████████| 201/201 [00:00<00:00, 20334.67it/s]
RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ✓ Cardiff RoBERTa


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 22764.59it/s]
RobertaForSequenceClassification LOAD REPORT from: siebert/sentiment-roberta-large-english
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ✓ SieBERT


...

  ✓ BERTweet


Loading weights: 100%|██████████| 394/394 [00:00<00:00, 3186.07it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: yangheng/deberta-v3-large-absa-v1.1
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ✓ DeBERTa ABSA


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 20706.28it/s]
RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ✓ Emotion Classifier

✓ All models loaded successfully!


## 8. Sentiment Analysis Helper Functions

In [10]:
def batch_sentiment_analysis(texts, model, batch_size=32, desc="Processing"):
    """Optimized batch sentiment analysis with progress tracking"""
    import time
    results = []
    start_time = time.time()
    
    # Convert to list and filter empty values
    texts_list = texts.fillna("").astype(str).values
    
    # Use tqdm if enabled
    iterator = tqdm(range(0, len(texts_list), batch_size), desc=desc, disable=not ENABLE_PROGRESS_BARS)
    
    for i in iterator:
        batch = texts_list[i:i+batch_size]
        
        try:
            batch_results = model(batch, truncation=True, batch_size=batch_size)
            for result in batch_results:
                results.append({
                    'label': result['label'],
                    'score': round(result['score'], 4)
                })
        except Exception as e:
            # Fallback to single processing on error
            for text in batch:
                try:
                    result = model(str(text).strip()[:512], truncation=True)[0]
                    results.append({
                        'label': result['label'],
                        'score': round(result['score'], 4)
                    })
                except:
                    results.append({'label': None, 'score': 0.0})
    
    elapsed = time.time() - start_time
    speed = len(texts_list) / elapsed
    print(f"  ✓ Processed {len(texts_list)} texts in {elapsed:.1f}s ({speed:.0f} texts/sec)")
    
    return pd.DataFrame(results)

print("✓ Batch sentiment analysis function defined")


✓ Batch sentiment analysis function defined


## 9. Apply Multi-Model Sentiment Analysis

In [11]:
# Step 1: Apply Cardiff RoBERTa to Title (OPTIMIZED BATCHING)
print("🔄 Analyzing Title sentiment (Cardiff RoBERTa) - BATCH MODE...")
title_results = batch_sentiment_analysis(df['Title'], cardiff_sentiment, batch_size=BATCH_SIZE, desc="Title sentiment")
df[['title_cardiff_sentiment', 'title_cardiff_score']] = title_results
print(f"  ✓ {df['title_cardiff_sentiment'].notna().sum()} rows processed")


🔄 Analyzing Title sentiment (Cardiff RoBERTa) - BATCH MODE...


Title sentiment: 100%|██████████| 505/505 [11:34<00:00,  1.38s/it]

  ✓ Processed 16156 texts in 694.8s (23 texts/sec)
  ✓ 16156 rows processed


In [12]:
# Step 2: Apply Cardiff RoBERTa to Text (OPTIMIZED BATCHING)
print("🔄 Analyzing Text sentiment (Cardiff RoBERTa) - BATCH MODE...")
text_results = batch_sentiment_analysis(df['Text'], cardiff_sentiment, batch_size=BATCH_SIZE, desc="Text sentiment")
df[['text_cardiff_sentiment', 'text_cardiff_score']] = text_results
print(f"  ✓ {df['text_cardiff_sentiment'].notna().sum()} rows processed")


🔄 Analyzing Text sentiment (Cardiff RoBERTa) - BATCH MODE...


Text sentiment: 100%|██████████| 505/505 [20:44<00:00,  2.47s/it]

  ✓ Processed 16156 texts in 1244.9s (13 texts/sec)
  ✓ 16156 rows processed


In [13]:
# Step 3: Combine Title + Text for comprehensive analysis
print("🔄 Creating combined Title+Text...")
df['combined_text'] = (df['Title'].fillna('') + ' ' + df['Text'].fillna('')).str.strip()
print(f"  ✓ Combined text created")

# Step 3a: Apply Cardiff RoBERTa to Combined (OPTIMIZED BATCHING)
print("🔄 Analyzing Combined sentiment (Cardiff RoBERTa) - BATCH MODE...")
combined_cardiff = batch_sentiment_analysis(df['combined_text'], cardiff_sentiment, batch_size=BATCH_SIZE, desc="Combined Cardiff")
df[['combined_cardiff_sentiment', 'combined_cardiff_score']] = combined_cardiff
print(f"  ✓ {df['combined_cardiff_sentiment'].notna().sum()} rows processed")

🔄 Creating combined Title+Text...
  ✓ Combined text created
🔄 Analyzing Combined sentiment (Cardiff RoBERTa) - BATCH MODE...


Combined Cardiff: 100%|██████████| 505/505 [21:33<00:00,  2.56s/it]

  ✓ Processed 16156 texts in 1293.6s (12 texts/sec)
  ✓ 16156 rows processed


In [ ]:
# Step 4: Apply SieBERT to Combined (OPTIMIZED BATCHING)
print("🔄 Analyzing Combined sentiment (SieBERT) - BATCH MODE...")
combined_siebert = batch_sentiment_analysis(df['combined_text'], siebert_sentiment, batch_size=BATCH_SIZE, desc="Combined SieBERT")
df[['combined_siebert_sentiment', 'combined_siebert_score']] = combined_siebert
print(f"  ✓ {df['combined_siebert_sentiment'].notna().sum()} rows processed")

🔄 Analyzing Combined sentiment (SieBERT) - BATCH MODE...


Combined SieBERT:  26%|██▌       | 131/505 [19:07<1:05:48, 10.56s/it]

In [ ]:
# Step 5: Apply BERTweet to Combined (OPTIMIZED BATCHING)
print("🔄 Analyzing Combined sentiment (BERTweet) - BATCH MODE...")
combined_bertweet = batch_sentiment_analysis(df['combined_text'], bertweet_sentiment, batch_size=BATCH_SIZE, desc="Combined BERTweet")
df[['combined_bertweet_sentiment', 'combined_bertweet_score']] = combined_bertweet
print(f"  ✓ {df['combined_bertweet_sentiment'].notna().sum()} rows processed")

In [ ]:
# Step 6: Apply DeBERTa ABSA to Combined (OPTIMIZED BATCHING)
print("🔄 Analyzing Combined sentiment (DeBERTa ABSA) - BATCH MODE...")
absa_results = batch_sentiment_analysis(df['combined_text'], absa_sentiment, batch_size=BATCH_SIZE, desc="Combined ABSA")
df[['absa_best_sentiment', 'absa_best_score']] = absa_results
print(f"  ✓ {df['absa_best_sentiment'].notna().sum()} rows processed")

In [ ]:
# Step 7: Apply Emotion Classifier to Text (OPTIMIZED BATCHING)
print("🔄 Analyzing Emotion (Text) - BATCH MODE...")
import time
start_time = time.time()

emotions = []
texts_list = df['Text'].fillna("").astype(str).values

for i in tqdm(range(0, len(texts_list), BATCH_SIZE), desc="Emotion detection", disable=not ENABLE_PROGRESS_BARS):
    batch = texts_list[i:i+BATCH_SIZE]
    try:
        batch_results = emotion_classifier(batch, truncation=True, batch_size=BATCH_SIZE, top_k=1)
        for result in batch_results:
            emotions.append(result[0]['label'] if result else None)
    except Exception:
        # Fallback to single processing
        for text in batch:
            try:
                result = emotion_classifier(str(text).strip()[:512], truncation=True, top_k=1)
                emotions.append(result[0]['label'] if result else None)
            except:
                emotions.append(None)

df['emotion'] = emotions
elapsed = time.time() - start_time
print(f"  ✓ Processed {len(texts_list)} texts in {elapsed:.1f}s ({len(texts_list)/elapsed:.0f} texts/sec)")
print(f"  ✓ {df['emotion'].notna().sum()} rows processed")


## 10. Ensemble Sentiment Voting

In [ ]:
# Create ensemble consensus from 4 models
def calculate_ensemble_sentiment(row):
    """Weighted voting from Cardiff, SieBERT, BERTweet, ABSA"""
    votes = []
    scores = []
    
    models = [
        (row['combined_cardiff_sentiment'], row['combined_cardiff_score']),
        (row['combined_siebert_sentiment'], row['combined_siebert_score']),
        (row['combined_bertweet_sentiment'], row['combined_bertweet_score']),
        (row['absa_best_sentiment'], row['absa_best_score']),
    ]
    
    for label, score in models:
        if pd.isna(label):
            continue
        
        norm_label = str(label).upper()
        if norm_label in ['NEGATIVE', 'NEG']:
            votes.append(-1)
        elif norm_label in ['POSITIVE', 'POS']:
            votes.append(1)
        else:
            votes.append(0)
        scores.append(score)
    
    if not votes:
        return 'NEUTRAL', 0.0
    
    weighted_sum = sum(v * s for v, s in zip(votes, scores))
    avg_score = sum(scores) / len(scores)
    
    if weighted_sum > 0.1:
        label = 'POSITIVE'
    elif weighted_sum < -0.1:
        label = 'NEGATIVE'
    else:
        label = 'NEUTRAL'
    
    return label, round(avg_score, 4)

print("🔄 Creating Ensemble Sentiment...")
df[['ensemble_sentiment', 'ensemble_score']] = df.apply(
    lambda row: pd.Series(calculate_ensemble_sentiment(row)), axis=1
)

print(f"✓ Ensemble created\n")
print("📊 Sentiment Distribution:")
print(df['ensemble_sentiment'].value_counts())
print(f"\n📈 Score Statistics:")
print(df['ensemble_score'].describe())

## 11. Feature Engineering: Text Measurements & Classifications

In [ ]:
# Classification features
print("🔄 Creating text measurements and classifications...")

# Rating classification
df['rating_class'] = pd.cut(df['Rating'], bins=[0, 2, 4, 5], 
                             labels=['Negative', 'Neutral', 'Positive'])

# Length buckets
df['length_bucket'] = pd.cut(df['word_count'], 
                             bins=[0, 50, 150, float('inf')],
                             labels=['Short', 'Medium', 'Long'])

# Sentiment-Rating Gap
def calculate_sentiment_rating_gap(row):
    """Calculate normalized gap between sentiment and rating"""
    try:
        if pd.isna(row['ensemble_sentiment']) or pd.isna(row['Rating']):
            return 0.0
        
        # Map sentiment to numeric scale
        sentiment_map = {'POSITIVE': 5, 'NEUTRAL': 3, 'NEGATIVE': 1}
        sentiment_value = sentiment_map.get(row['ensemble_sentiment'], 3)
        
        rating = float(row['Rating'])
        gap = sentiment_value - rating
        
        return round(gap / 4, 4)  # Normalize to -1 to 1
    except:
        return 0.0

df['sentiment_rating_gap'] = df.apply(calculate_sentiment_rating_gap, axis=1)

# Rating-Sentiment match
df['rating_sentiment_match'] = (
    ((df['Rating'] >= 4) & (df['ensemble_sentiment'] == 'POSITIVE')) |
    ((df['Rating'] < 3) & (df['ensemble_sentiment'] == 'NEGATIVE')) |
    ((df['Rating'] == 3) & (df['ensemble_sentiment'] == 'NEUTRAL'))
).astype(int)

df['inconsistency_flag'] = (~df['rating_sentiment_match'].astype(bool)).astype(int)

print(f"✓ Text measurements created")
print(f"  Rating classes: {df['rating_class'].value_counts().to_dict()}")
print(f"  Length buckets: {df['length_bucket'].value_counts().to_dict()}")
print(f"  Sentiment-Rating mismatch: {df['inconsistency_flag'].sum()} reviews")


## 12. Topic Modeling with BERTopic

In [ ]:
# Install BERTopic if needed
try:
    from bertopic import BERTopic
    from sklearn.feature_extraction.text import CountVectorizer
except ImportError:
    print("Installing BERTopic and dependencies...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "bertopic", "sentence-transformers"])
    from bertopic import BERTopic
    from sklearn.feature_extraction.text import CountVectorizer

# Prepare clean text for topic modeling
print("🔄 Preparing text for topic modeling...")
df['text_for_topics'] = (df['Title'].fillna('') + ' ' + df['Text'].fillna('')).str.strip()

# Filter out very short texts and apply sampling
valid_docs = df[df['text_for_topics'].str.len() > 20].copy()
print(f"  Valid reviews: {len(valid_docs)} (min 20 chars)")

# Apply sampling if SAMPLE_SIZE is set
if SAMPLE_SIZE and len(valid_docs) > SAMPLE_SIZE:
    print(f"  Sampling {SAMPLE_SIZE} documents for faster processing...")
    valid_docs = valid_docs.sample(n=SAMPLE_SIZE, random_state=42)
    print(f"  Using {len(valid_docs)} samples for topic modeling")

# Initialize BERTopic with optimizations
print("🔄 Training BERTopic model (OPTIMIZED)...")
start_time = time.time()

try:
    topic_model = BERTopic(
        language="english",
        calculate_probabilities=True,
        min_topic_size=5,
        nr_topics=None,
        n_gram_range=(1, 2),
        top_n_words=10,
        diversity=0.5,
        verbose=False,  # Disable verbose output for cleaner logs
    )
    
    topics, probabilities = topic_model.fit_transform(valid_docs['text_for_topics'].values)
    elapsed = time.time() - start_time
    
    # Get topic info
    topic_df = topic_model.get_topic_freq()
    print(f"✓ Model trained in {elapsed:.1f}s")
    print(f"\n✓ Topics identified: {len(topic_df)}")
    print(f"\n📊 Top 10 Topics by frequency:")
    print(topic_df.head(10))
    
    # Map topics back to full dataframe
    df['dominant_topic'] = -1  # Default for excluded rows
    df.loc[valid_docs.index, 'dominant_topic'] = topics
    df['topic_probability'] = 0.0
    df.loc[valid_docs.index, 'topic_probability'] = probabilities.max(axis=1)
    
    # Get topic keywords for each row (OPTIMIZED - vectorized where possible)
    def get_topic_keywords(topic_id):
        if topic_id == -1:
            return ""
        try:
            topic_words = topic_model.get_topic(topic_id)
            if topic_words:
                return ", ".join([word for word, score in topic_words[:5]])
        except:
            pass
        return ""
    
    df['topic_keywords'] = df['dominant_topic'].apply(get_topic_keywords)
    
    print(f"\n✓ Topic assignments complete")
    print(f"  Reviews with topics: {(df['dominant_topic'] != -1).sum()}")
    print(f"  Avg topic probability: {df.loc[df['dominant_topic'] != -1, 'topic_probability'].mean():.3f}")
    
except Exception as e:
    print(f"⚠️  BERTopic failed: {str(e)}")
    print("  Creating placeholder columns...")
    df['dominant_topic'] = -1
    df['topic_probability'] = 0.0
    df['topic_keywords'] = ""


## 12b. KeyBERT: Aspect-Level Keyword Extraction (OPTIMIZED)

In [ ]:
# Install KeyBERT if needed
try:
    from keybert import KeyBERT
except ImportError:
    print("Installing KeyBERT...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "keybert"])
    from keybert import KeyBERT

print("🔄 Initializing KeyBERT for aspect extraction...")
try:
    kw_model = KeyBERT(model='sentence-transformers/all-MiniLM-L6-v2', language="english")
    
    # Extract keywords from sample (faster) or all reviews
    sample_for_extraction = valid_docs.copy() if len(valid_docs) > SAMPLE_SIZE else df.copy()
    
    print(f"🔄 Extracting keywords from {len(sample_for_extraction)} reviews - BATCH MODE...")
    start_time = time.time()
    
    keywords_list = []
    
    for i in tqdm(range(0, len(sample_for_extraction), BATCH_SIZE), desc="KeyBERT extraction", disable=not ENABLE_PROGRESS_BARS):
        batch_texts = sample_for_extraction['text_for_topics'].iloc[i:i+BATCH_SIZE].values
        
        for text in batch_texts:
            try:
                keywords = kw_model.extract_keywords(
                    text, 
                    language='english',
                    top_n=5,
                    use_maxsum=True,
                    nr_candidates=20
                )
                kw_str = ", ".join([kw for kw, score in keywords])
                keywords_list.append(kw_str)
            except Exception as e:
                keywords_list.append("")
    
    # Map back to dataframe indices
    df['extracted_keywords'] = ""
    df.loc[sample_for_extraction.index, 'extracted_keywords'] = keywords_list
    
    elapsed = time.time() - start_time
    print(f"✓ Extracted keywords in {elapsed:.1f}s ({len(sample_for_extraction)/elapsed:.0f} reviews/sec)")
    print(f"  ✓ Reviews with keywords: {(df['extracted_keywords'] != '').sum()}")
    
except Exception as e:
    print(f"⚠️  KeyBERT failed: {str(e)}")
    print("  Creating placeholder column...")
    df['extracted_keywords'] = ""

print("\n✓ Aspect extraction complete")


## 13. Reviewer Behavior & Helpfulness Analysis

In [ ]:
print("🔄 Computing reviewer behavior and helpfulness metrics...")

# Helpfulness features
df['has_helpful_votes'] = (df['Helpful_Votes'] > 0).astype(int)

# Helpful vote buckets
df['helpful_vote_bucket'] = pd.cut(df['Helpful_Votes'], 
                                  bins=[-0.1, 0, 10, 50, float('inf')],
                                  labels=['None', 'Low', 'Medium', 'High'])

# Reviewer experience level (contribution-based)
if 'User_Contributions' in df.columns:
    df['reviewer_experience_level'] = pd.cut(df['User_Contributions'], 
                                            bins=[-1, 5, 50, float('inf')],
                                            labels=['Low', 'Medium', 'High'])
else:
    df['reviewer_experience_level'] = 'Unknown'

# Review quality score (composite of multiple factors)
def calculate_quality_score(row):
    """Composite quality metric"""
    score = 0.0
    
    # Length factor (0.2)
    if row['word_count'] > 100:
        score += 0.2
    elif row['word_count'] > 50:
        score += 0.1
    
    # Helpfulness factor (0.3)
    if row['has_helpful_votes']:
        score += 0.15
    if row['Helpful_Votes'] > 50:
        score += 0.15
    
    # Consistency factor (0.2)
    if row['rating_sentiment_match']:
        score += 0.2
    
    # Sentiment clarity factor (0.3)
    if row['ensemble_score'] > 0.8:
        score += 0.3
    elif row['ensemble_score'] > 0.6:
        score += 0.15
    
    return round(score, 3)

df['review_quality_score'] = df.apply(calculate_quality_score, axis=1)

# Helpfulness ratio
df['helpfulness_ratio'] = (df['Helpful_Votes'] / (df['word_count'] + 1)).round(4)

print(f"✓ Helpfulness analysis complete")
print(f"  Reviews with helpful votes: {df['has_helpful_votes'].sum()}")
print(f"  Avg quality score: {df['review_quality_score'].mean():.3f}")
print(f"  Avg helpfulness ratio: {df['helpfulness_ratio'].mean():.4f}")


## 14. Destination Performance Aggregates

In [ ]:
print("🔄 Computing destination performance metrics...")

# Location aggregates
def encode_sentiment(sentiment):
    """Convert sentiment to numeric"""
    return {'POSITIVE': 1.0, 'NEUTRAL': 0.5, 'NEGATIVE': 0.0}.get(sentiment, 0.5)

df['sentiment_numeric'] = df['ensemble_sentiment'].apply(encode_sentiment)

# By location (if available)
if 'Location_Name' in df.columns:
    location_stats = df.groupby('Location_Name').agg({
        'Rating': ['mean', 'count', 'std'],
        'sentiment_numeric': 'mean',
        'ensemble_sentiment': lambda x: (x == 'POSITIVE').sum() / len(x)
    }).round(3)
    
    # Map back to dataframe
    df['destination_avg_rating'] = df['Location_Name'].map(
        df.groupby('Location_Name')['Rating'].mean()
    )
    df['destination_review_count'] = df['Location_Name'].map(
        df.groupby('Location_Name').size()
    )
    df['destination_sentiment_mean'] = df['Location_Name'].map(
        df.groupby('Location_Name')['sentiment_numeric'].mean()
    )
else:
    # By city as fallback
    df['destination_avg_rating'] = df['Located_City'].map(
        df.groupby('Located_City')['Rating'].mean()
    )
    df['destination_review_count'] = df['Located_City'].map(
        df.groupby('Located_City').size()
    )
    df['destination_sentiment_mean'] = df['Located_City'].map(
        df.groupby('Located_City')['sentiment_numeric'].mean()
    )

# By province (using province from location extraction)
if 'province' in df.columns:
    df['province_avg_rating'] = df['province'].map(
        df.groupby('province')['Rating'].mean()
    )
    df['province_review_count'] = df['province'].map(
        df.groupby('province').size()
    )

# Ranking features
destination_col = 'Location_Name' if 'Location_Name' in df.columns else 'Located_City'
rating_rank = df.groupby(destination_col)['Rating'].mean().rank(ascending=False)
review_rank = df.groupby(destination_col).size().rank(ascending=False)

df['destination_rank_by_rating'] = df[destination_col].map(rating_rank).astype('Int64')
df['destination_rank_by_reviews'] = df[destination_col].map(review_rank).astype('Int64')

# Quality vs Popularity gap
df['popularity_vs_quality_gap'] = (
    df['destination_rank_by_reviews'] - df['destination_rank_by_rating']
).fillna(0).astype('Int64')

print(f"✓ Destination metrics computed")
print(f"\n📊 Top 10 destinations by rating:")
top_by_rating = df.groupby(destination_col)['Rating'].agg(['mean', 'count']).sort_values('mean', ascending=False).head(10)
print(top_by_rating)


## 15. Analysis 1: Descriptive Analysis

In [ ]:
print("=" * 80)
print("Analysis 1: DESCRIPTIVE ANALYSIS")
print("=" * 80)

# Dataset overview
print("\n📊 DATASET OVERVIEW")
print(f"  Total reviews: {len(df):,}")
print(f"  Date range: {df['Travel_Date'].min().date()} to {df['Travel_Date'].max().date()}")
print(f"  Total columns: {len(df.columns)}")

# Location distribution
print("\n📍 LOCATION DISTRIBUTION")
destination_col = 'Location_Name' if 'Location_Name' in df.columns else 'Located_City'
print(f"  Unique destinations: {df[destination_col].nunique()}")
print(f"  Reviews per destination: {df.groupby(destination_col).size().describe()}")

if 'province' in df.columns:
    print(f"\n  Unique provinces: {df['province'].nunique()}")
    print(f"  Provincial distribution:\n{df['province'].value_counts()}")

# Tourist origin
print("\n🌍 TOURIST ORIGIN")
print(f"  Unique countries: {df['user_country'].nunique()}")
print(f"  Top 15 countries:\n{df['user_country'].value_counts().head(15)}")

# Rating distribution
print("\n⭐ RATING DISTRIBUTION")
print(df['Rating'].describe())
print(f"\n  Distribution:\n{df['Rating'].value_counts().sort_index()}")

# Review activity
print("\n📅 REVIEW ACTIVITY")
print(f"  Reviews per year:\n{df['Published_Date_Year'].value_counts().sort_index()}")
print(f"  Peak travel month: {df['Travel_Date_Month'].mode()[0]}")
print(f"  Peak review month: {df['Published_Date_Month'].mode()[0]}")


## 16. Analysis 2: Tourist Satisfaction Analysis

In [ ]:
print("\n" + "=" * 80)
print("Analysis 2: TOURIST SATISFACTION ANALYSIS")
print("=" * 80)

destination_col = 'Location_Name' if 'Location_Name' in df.columns else 'Located_City'

# Overall satisfaction
print("\n🎯 OVERALL SATISFACTION")
print(f"  Average rating: {df['Rating'].mean():.2f}/5.0")
print(f"  Rating class distribution:\n{df['rating_class'].value_counts()}")

# Destination satisfaction
print(f"\n📍 TOP 15 DESTINATIONS BY RATING")
top_dest = df.groupby(destination_col).agg({
    'Rating': ['mean', 'count', 'std'],
    'ensemble_sentiment': lambda x: f"{(x == 'POSITIVE').sum() / len(x) * 100:.1f}%"
}).round(2)
top_dest.columns = ['Avg_Rating', 'Review_Count', 'Std_Dev', 'Positive_%']
top_dest = top_dest.sort_values('Avg_Rating', ascending=False).head(15)
print(top_dest)

# Worst destinations
print(f"\n⚠️  BOTTOM 15 DESTINATIONS BY RATING")
bottom_dest = df.groupby(destination_col).agg({
    'Rating': ['mean', 'count'],
    'ensemble_sentiment': lambda x: f"{(x == 'POSITIVE').sum() / len(x) * 100:.1f}%"
}).round(2)
bottom_dest.columns = ['Avg_Rating', 'Review_Count', 'Positive_%']
bottom_dest = bottom_dest.sort_values('Avg_Rating', ascending=True).head(15)
print(bottom_dest)

# Local vs International
if 'user_country' in df.columns:
    print(f"\n🌍 DOMESTIC VS INTERNATIONAL SATISFACTION")
    local_ratings = df[df['user_country'].isna()]['Rating'].mean()
    intl_ratings = df[df['user_country'].notna()]['Rating'].mean()
    print(f"  Local reviews avg: {local_ratings:.2f}")
    print(f"  International reviews avg: {intl_ratings:.2f}")

# Top countries' satisfaction
print(f"\n🌎 TOP 10 COUNTRIES - SATISFACTION LEVELS")
country_sat = df[df['user_country'].notna()].groupby('user_country').agg({
    'Rating': ['mean', 'count']
}).round(2)
country_sat.columns = ['Avg_Rating', 'Review_Count']
country_sat = country_sat[country_sat['Review_Count'] >= 5].sort_values('Avg_Rating', ascending=False).head(10)
print(country_sat)

# Province-level satisfaction
if 'province' in df.columns:
    print(f"\n🏛️  PROVINCE-LEVEL SATISFACTION")
    province_sat = df.groupby('province').agg({
        'Rating': ['mean', 'count']
    }).round(2)
    province_sat.columns = ['Avg_Rating', 'Review_Count']
    province_sat = province_sat.sort_values('Avg_Rating', ascending=False)
    print(province_sat)


## 17. Analysis 3: Sentiment Distribution Analysis

In [ ]:
print("\n" + "=" * 80)
print("Analysis 3: SENTIMENT DISTRIBUTION ANALYSIS")
print("=" * 80)

destination_col = 'Location_Name' if 'Location_Name' in df.columns else 'Located_City'

# Overall sentiment
print("\n😊 OVERALL SENTIMENT DISTRIBUTION")
sentiment_dist = df['ensemble_sentiment'].value_counts()
print(f"  {sentiment_dist}")

# Sentiment by rating
print("\n📊 SENTIMENT BY RATING CLASS")
rating_sentiment = pd.crosstab(df['rating_class'], df['ensemble_sentiment'], margins=True)
print(rating_sentiment)

# Sentiment by destination
print(f"\n📍 TOP 10 DESTINATIONS - SENTIMENT BREAKDOWN")
dest_sentiment = df.groupby(destination_col)['ensemble_sentiment'].value_counts().unstack(fill_value=0)
dest_sentiment['Total'] = dest_sentiment.sum(axis=1)
dest_sentiment['Positive%'] = (dest_sentiment['POSITIVE'] / dest_sentiment['Total'] * 100).round(1)
dest_sentiment = dest_sentiment.sort_values('Positive%', ascending=False).head(10)
print(dest_sentiment)

# Emotion distribution
print("\n💭 EMOTION DISTRIBUTION")
emotion_dist = df['emotion'].value_counts()
print(emotion_dist)
print(f"  Positive emotions (joy): {emotion_dist.get('joy', 0)}")
print(f"  Negative emotions (anger, sadness, fear): {emotion_dist.get('anger', 0) + emotion_dist.get('sadness', 0) + emotion_dist.get('fear', 0)}")

# Emotion by destination
if df['emotion'].notna().any():
    print(f"\n💭 DOMINANT EMOTION BY TOP DESTINATIONS")
    dest_emotion = df.groupby(destination_col)['emotion'].apply(lambda x: x.mode()[0] if len(x.mode()) > 0 else 'Unknown')
    print(dest_emotion.head(10))

# Sentiment confidence
print(f"\n🎯 SENTIMENT CONFIDENCE (Ensemble Score)")
print(f"  Mean confidence: {df['ensemble_score'].mean():.3f}")
print(f"  High confidence (>0.8): {(df['ensemble_score'] > 0.8).sum()} reviews ({(df['ensemble_score'] > 0.8).sum() / len(df) * 100:.1f}%)")
print(f"  Low confidence (<0.6): {(df['ensemble_score'] < 0.6).sum()} reviews ({(df['ensemble_score'] < 0.6).sum() / len(df) * 100:.1f}%)")

# Model agreement
print(f"\n🤝 MODEL AGREEMENT ANALYSIS")
sentiment_cols = ['combined_cardiff_sentiment', 'combined_siebert_sentiment', 'combined_bertweet_sentiment', 'absa_best_sentiment']
valid_cols = [col for col in sentiment_cols if col in df.columns]

if valid_cols:
    agreement_rate = (df[valid_cols[0]] == df[valid_cols[1]]).mean() if len(valid_cols) > 1 else 0
    print(f"  Model agreement rate: {agreement_rate * 100:.1f}%")
    print(f"  Models compared: {', '.join([col.replace('combined_', '').replace('_sentiment', '') for col in valid_cols[:2]])}")


## 18. Analysis 4: Topic Modeling Insights

In [ ]:
print("\n" + "=" * 80)
print("Analysis 4: TOPIC MODELING INSIGHTS")
print("=" * 80)

if 'dominant_topic' in df.columns and df['dominant_topic'].max() > -1:
    print(f"\n🧠 TOPIC DISTRIBUTION")
    topic_dist = df['dominant_topic'].value_counts()
    print(f"  Total topics identified: {len(topic_dist)}")
    print(f"  Reviews with topics: {(df['dominant_topic'] != -1).sum()}")
    print(f"  Avg topic probability: {df['topic_probability'].mean():.3f}")
    
    print(f"\n🔝 TOP 10 TOPICS BY FREQUENCY")
    print(topic_dist.head(10))
    
    # Topic-sentiment relationship
    print(f"\n😊 SENTIMENT BY TOPIC")
    if (df['dominant_topic'] != -1).any():
        topic_sentiment = df[df['dominant_topic'] != -1].groupby('dominant_topic').agg({
            'ensemble_sentiment': lambda x: f"{(x == 'POSITIVE').sum() / len(x) * 100:.1f}%",
            'Rating': 'mean'
        }).round(2)
        topic_sentiment.columns = ['Positive%', 'Avg_Rating']
        topic_sentiment = topic_sentiment.sort_values('Positive%', ascending=False).head(10)
        print(topic_sentiment)
    
    # Sample keywords from top topics
    print(f"\n🏷️  SAMPLE KEYWORDS FROM TOP TOPICS")
    for topic_id in df['dominant_topic'].value_counts().head(5).index:
        keywords = df[df['dominant_topic'] == topic_id]['topic_keywords'].iloc[0]
        print(f"  Topic {topic_id}: {keywords}")
else:
    print("⚠️  Topic modeling not available or not executed")


## 19. Analysis 5: Rating-Sentiment Consistency & Quality Flags

In [ ]:
print("\n" + "=" * 80)
print("Analysis 5: RATING-SENTIMENT CONSISTENCY & QUALITY FLAGS")
print("=" * 80)

# Consistency overview
print("\n✅ RATING-SENTIMENT ALIGNMENT")
match_pct = df['rating_sentiment_match'].mean() * 100
print(f"  Consistent reviews: {df['rating_sentiment_match'].sum()} ({match_pct:.1f}%)")
print(f"  Inconsistent reviews: {df['inconsistency_flag'].sum()} ({100 - match_pct:.1f}%)")

# Sentiment-rating gap analysis
print(f"\n🔄 SENTIMENT-RATING GAP")
print(f"  Mean gap: {df['sentiment_rating_gap'].mean():.3f}")
print(f"  Gap distribution:")
print(f"    Gap < -0.5 (Negative text, High rating): {(df['sentiment_rating_gap'] < -0.5).sum()}")
print(f"    Gap > +0.5 (Positive text, Low rating): {(df['sentiment_rating_gap'] > 0.5).sum()}")
print(f"    Gap near 0 (Consistent): {((df['sentiment_rating_gap'] >= -0.25) & (df['sentiment_rating_gap'] <= 0.25)).sum()}")

# Title vs Body consistency
if 'title_cardiff_sentiment' in df.columns and 'text_cardiff_sentiment' in df.columns:
    title_text_match = (df['title_cardiff_sentiment'] == df['text_cardiff_sentiment']).sum()
    title_text_match_pct = title_text_match / len(df) * 100
    print(f"\n📋 TITLE-TEXT SENTIMENT ALIGNMENT")
    print(f"  Matching title/text: {title_text_match} ({title_text_match_pct:.1f}%)")
    
    mismatches = df[df['title_cardiff_sentiment'] != df['text_cardiff_sentiment']]
    if len(mismatches) > 0:
        print(f"\n  Mismatch types:")
        print(mismatches[['title_cardiff_sentiment', 'text_cardiff_sentiment']].value_counts().head(5))

# Suspicious reviews (inconsistency flags)
print(f"\n🚩 QUALITY FLAGS & SUSPICIOUS REVIEWS")
print(f"  Low confidence (<0.6): {(df['ensemble_score'] < 0.6).sum()}")
print(f"  Rating-sentiment mismatch: {df['inconsistency_flag'].sum()}")
print(f"  No helpful votes + negative: {((df['has_helpful_votes'] == 0) & (df['ensemble_sentiment'] == 'NEGATIVE')).sum()}")

suspicious = df[
    (df['inconsistency_flag'] == 1) | 
    (df['ensemble_score'] < 0.5)
].shape[0]
print(f"  ⚠️  Total flagged for review: {suspicious}")

# Sample inconsistencies
if (df['inconsistency_flag'] == 1).any():
    print(f"\n  Sample inconsistent reviews:")
    samples = df[df['inconsistency_flag'] == 1][['Rating', 'ensemble_sentiment', 'sentiment_rating_gap', 'word_count']].head(5)
    print(samples)


## 20. Analysis 6: Temporal & Seasonal Patterns

In [ ]:
print("\n" + "=" * 80)
print("Analysis 6: TEMPORAL & SEASONAL PATTERNS")
print("=" * 80)

# Review lag analysis
df['review_delay_days'] = (df['Published_Date'] - df['Travel_Date']).dt.days
df['review_delay_days'] = df['review_delay_days'].clip(lower=0)  # Remove negative values

print("\n⏱️  REVIEW PUBLICATION LAG")
print(f"  Mean delay: {df['review_delay_days'].mean():.0f} days")
print(f"  Median delay: {df['review_delay_days'].median():.0f} days")
print(f"  Max delay: {df['review_delay_days'].max():.0f} days")

# Delay impact on sentiment
delay_categories = pd.cut(df['review_delay_days'], bins=[0, 30, 90, 180, float('inf')],
                          labels=['<1 month', '1-3 months', '3-6 months', '>6 months'])
print(f"\n  Sentiment by review delay:")
delay_sentiment = df.groupby(delay_categories)['Rating'].mean()
print(delay_sentiment)

# Seasonal patterns
print(f"\n🗓️  SEASONAL SATISFACTION PATTERNS")
season_rating = df.groupby('Travel_Date_Month')['Rating'].agg(['mean', 'count']).round(2)
season_rating.columns = ['Avg_Rating', 'Reviews']
print(season_rating)

# Year-over-year trends
if df['Travel_Date_Year'].nunique() > 1:
    print(f"\n📈 YEAR-OVER-YEAR TRENDS")
    yearly_rating = df.groupby('Travel_Date_Year')['Rating'].agg(['mean', 'count']).round(2)
    yearly_rating.columns = ['Avg_Rating', 'Review_Count']
    yearly_rating = yearly_rating[yearly_rating['Review_Count'] >= 5]  # Only years with 5+ reviews
    print(yearly_rating)

# Review publication trends
print(f"\n📅 REVIEWS BY PUBLICATION YEAR")
publish_yearly = df.groupby('Published_Date_Year')['Rating'].agg(['mean', 'count']).round(2)
publish_yearly.columns = ['Avg_Rating', 'Review_Count']
print(publish_yearly.tail(10))

# Peak seasons
print(f"\n🏝️  PEAK TRAVEL MONTHS")
peak_months = {1: 'January', 2: 'February', 3: 'March', 4: 'April', 5: 'May', 6: 'June',
               7: 'July', 8: 'August', 9: 'September', 10: 'October', 11: 'November', 12: 'December'}
monthly_counts = df['Travel_Date_Month'].value_counts().sort_index()
print(f"  Peak month: {peak_months[monthly_counts.idxmax()]} ({monthly_counts.max()} reviews)")
print(f"  Slowest month: {peak_months[monthly_counts.idxmin()]} ({monthly_counts.min()} reviews)")


## 21. Analysis 7: Review Behavior & Quality Analysis

In [ ]:
print("\n" + "=" * 80)
print("Analysis 7: REVIEW BEHAVIOR & QUALITY ANALYSIS")
print("=" * 80)

# Text length analysis
print("\n📏 REVIEW LENGTH ANALYSIS")
print(f"  Avg review length: {df['review_length'].mean():.0f} characters")
print(f"  Avg word count: {df['word_count'].mean():.0f} words")
print(f"  Avg title length: {df['title_length'].mean():.0f} characters")

print(f"\n  Length distribution:")
length_dist = df['length_bucket'].value_counts()
print(length_dist)

# Length vs satisfaction
print(f"\n⭐ LENGTH IMPACT ON SATISFACTION")
length_rating = df.groupby('length_bucket')['Rating'].agg(['mean', 'count']).round(2)
length_rating.columns = ['Avg_Rating', 'Review_Count']
print(length_rating)

# Length vs sentiment
print(f"\n  Sentiment by review length:")
length_sentiment = df.groupby('length_bucket')['ensemble_sentiment'].value_counts().unstack(fill_value=0)
print(length_sentiment)

# Reviewer experience level
if 'reviewer_experience_level' in df.columns:
    print(f"\n👤 REVIEWER EXPERIENCE LEVELS")
    if df['reviewer_experience_level'].dtype == 'object' and df['reviewer_experience_level'].iloc[0] != 'Unknown':
        exp_dist = df['reviewer_experience_level'].value_counts()
        print(exp_dist)
        
        # Experience level vs quality
        print(f"\n  Satisfaction by reviewer experience:")
        exp_rating = df.groupby('reviewer_experience_level')['Rating'].agg(['mean', 'count']).round(2)
        exp_rating.columns = ['Avg_Rating', 'Reviews']
        print(exp_rating)

# Quality score distribution
print(f"\n🎯 REVIEW QUALITY SCORES")
print(f"  Mean quality score: {df['review_quality_score'].mean():.3f}")
print(f"  Distribution:")
quality_buckets = pd.cut(df['review_quality_score'], bins=[0, 0.3, 0.6, 0.9, 1.0],
                         labels=['Low', 'Medium', 'High', 'Very High'])
print(quality_buckets.value_counts().sort_index())

print(f"\n  Quality score vs Rating:")
quality_rating = df.groupby(quality_buckets)['Rating'].agg(['mean', 'count']).round(2)
quality_rating.columns = ['Avg_Rating', 'Reviews']
print(quality_rating)


## 22. Analysis 8: Helpfulness & Impact Analysis

In [ ]:
print("\n" + "=" * 80)
print("Analysis 8: HELPFULNESS & IMPACT ANALYSIS")
print("=" * 80)

# Helpful votes overview
print("\n👍 HELPFUL VOTES OVERVIEW")
print(f"  Total helpful votes: {df['Helpful_Votes'].sum():,}")
print(f"  Avg votes per review: {df['Helpful_Votes'].mean():.1f}")
print(f"  Reviews with votes: {df['has_helpful_votes'].sum()} ({df['has_helpful_votes'].mean() * 100:.1f}%)")

# Helpful vote distribution
print(f"\n  Reviews by helpful vote bucket:")
helpful_dist = df['helpful_vote_bucket'].value_counts()
print(helpful_dist)

# What makes reviews helpful?
print(f"\n📊 FACTORS ASSOCIATED WITH HELPFUL VOTES")

# Length and helpfulness
length_helpful = df.groupby('length_bucket')['Helpful_Votes'].agg(['sum', 'mean', 'count']).round(1)
length_helpful.columns = ['Total_Votes', 'Avg_Votes', 'Reviews']
print(f"\n  By review length:")
print(length_helpful)

# Rating and helpfulness
rating_class_helpful = df.groupby('rating_class')['Helpful_Votes'].agg(['sum', 'mean', 'count']).round(1)
rating_class_helpful.columns = ['Total_Votes', 'Avg_Votes', 'Reviews']
print(f"\n  By rating class:")
print(rating_class_helpful)

# Sentiment and helpfulness
sentiment_helpful = df.groupby('ensemble_sentiment')['Helpful_Votes'].agg(['sum', 'mean', 'count']).round(1)
sentiment_helpful.columns = ['Total_Votes', 'Avg_Votes', 'Reviews']
print(f"\n  By sentiment:")
print(sentiment_helpful)

# Quality score and helpfulness
quality_helpful = df.groupby(quality_buckets)['Helpful_Votes'].agg(['sum', 'mean', 'count']).round(1)
quality_helpful.columns = ['Total_Votes', 'Avg_Votes', 'Reviews']
print(f"\n  By quality score:")
print(quality_helpful)

# Correlation analysis
print(f"\n📈 HELPFULNESS CORRELATIONS")
corr_with_helpful = df[['Helpful_Votes', 'word_count', 'Rating', 'review_quality_score', 'ensemble_score']].corr()['Helpful_Votes'].sort_values(ascending=False)
print(corr_with_helpful)

# Top helpful reviews insights
print(f"\n⭐ TOP HELPFUL REVIEWS CHARACTERISTICS")
top_helpful = df.nlargest(10, 'Helpful_Votes')[['Rating', 'word_count', 'ensemble_sentiment', 'Helpful_Votes']].head()
print(f"  Avg rating of top 10: {top_helpful['Rating'].mean():.2f}")
print(f"  Avg length of top 10: {top_helpful['word_count'].mean():.0f} words")
print(f"  Sentiment breakdown of top 10:")
print(top_helpful['ensemble_sentiment'].value_counts())


## 23. Analysis 9: Geographic & Tourist Origin Analysis

In [ ]:
print("\n" + "=" * 80)
print("Analysis 9: GEOGRAPHIC & TOURIST ORIGIN ANALYSIS")
print("=" * 80)

# Province analysis (if available)
if 'province' in df.columns and df['province'].notna().any():
    print("\n🏛️  DESTINATION PROVINCE ANALYSIS")
    province_analysis = df.groupby('province').agg({
        'Rating': ['mean', 'count', 'std'],
        'ensemble_sentiment': lambda x: f"{(x == 'POSITIVE').sum() / len(x) * 100:.1f}%"
    }).round(2)
    province_analysis.columns = ['Avg_Rating', 'Count', 'Std', 'Positive%']
    province_analysis = province_analysis.sort_values('Avg_Rating', ascending=False)
    print(province_analysis)

# District analysis (if available)
if 'district' in df.columns and df['district'].notna().any():
    print(f"\n📍 TOP DISTRICTS BY RATING")
    district_analysis = df.groupby('district').agg({
        'Rating': ['mean', 'count']
    }).round(2)
    district_analysis.columns = ['Avg_Rating', 'Review_Count']
    district_analysis = district_analysis[district_analysis['Review_Count'] >= 3]  # At least 3 reviews
    district_analysis = district_analysis.sort_values('Avg_Rating', ascending=False).head(15)
    print(district_analysis)

# Tourist origin analysis
print("\n🌍 TOURIST ORIGIN ANALYSIS")
print(f"  Countries represented: {df['user_country'].nunique()}")
print(f"  Top 20 source countries:")
top_countries = df['user_country'].value_counts().head(20)
print(top_countries)

# Satisfaction by origin
print(f"\n⭐ SATISFACTION BY TOURIST ORIGIN (Top 15 countries)")
country_satisfaction = df[df['user_country'].notna()].groupby('user_country').agg({
    'Rating': ['mean', 'count']
}).round(2)
country_satisfaction.columns = ['Avg_Rating', 'Review_Count']
country_satisfaction = country_satisfaction[country_satisfaction['Review_Count'] >= 5]
country_satisfaction = country_satisfaction.sort_values('Avg_Rating', ascending=False).head(15)
print(country_satisfaction)

# Regional patterns
if 'user_region' in df.columns:
    print(f"\n🗺️  REGIONAL PATTERNS (for countries with regions)")
    regions_with_data = df[df['user_region'].notna()]['user_region'].nunique()
    if regions_with_data > 0:
        print(f"  Unique regions identified: {regions_with_data}")

# Origin-destination heatmap data
print(f"\n🔥 ORIGIN-DESTINATION PATTERNS")
destination_col = 'Location_Name' if 'Location_Name' in df.columns else 'Located_City'
if (df['user_country'].notna().sum() > 0) and (df[destination_col].notna().sum() > 0):
    # Show top country-destination combinations
    top_pairs = df.groupby([destination_col, 'user_country']).size().sort_values(ascending=False).head(10)
    print(f"  Top 10 Country-Destination pairs:")
    for (dest, country), count in top_pairs.items():
        print(f"    {country} → {dest}: {count} reviews")


## 24. Analysis 10: Destination Rankings & Winners/Losers

In [ ]:
print("\n" + "=" * 80)
print("Analysis 10: DESTINATION RANKINGS & PERFORMANCE GAPS")
print("=" * 80)

destination_col = 'Location_Name' if 'Location_Name' in df.columns else 'Located_City'

# Quality winners (high rating, many reviews)
print("\n🏆 QUALITY WINNERS (High Rating + High Volume)")
quality_winners = df.groupby(destination_col).agg({
    'Rating': ['mean', 'count'],
    'ensemble_sentiment': lambda x: (x == 'POSITIVE').sum() / len(x) * 100
}).round(2)
quality_winners.columns = ['Avg_Rating', 'Review_Count', 'Positive%']
quality_winners = quality_winners[quality_winners['Review_Count'] >= 10]  # At least 10 reviews
quality_winners = quality_winners.sort_values(['Avg_Rating', 'Review_Count'], ascending=[False, False])
print(quality_winners.head(15))

# Popularity leaders (most reviewed)
print(f"\n📈 POPULARITY LEADERS (Most Reviews)")
popularity = df.groupby(destination_col)['Rating'].count().sort_values(ascending=False)
popularity = popularity[popularity >= 5]  # At least 5 reviews
popular_df = df.groupby(destination_col).agg({
    'Rating': ['mean', 'count'],
    'ensemble_sentiment': lambda x: (x == 'POSITIVE').sum() / len(x) * 100
}).round(2)
popular_df.columns = ['Avg_Rating', 'Review_Count', 'Positive%']
popular_df = popular_df.loc[popularity.index[:15]]
print(popular_df)

# Rising stars (high rating, lower volume - emerging destinations)
print(f"\n⭐ RISING STARS (High Rating, Lower Volume)")
rising_stars = df.groupby(destination_col).agg({
    'Rating': ['mean', 'count']
}).round(2)
rising_stars.columns = ['Avg_Rating', 'Review_Count']
rising_stars = rising_stars[(rising_stars['Review_Count'] >= 3) & (rising_stars['Review_Count'] < 10)]
rising_stars = rising_stars.sort_values('Avg_Rating', ascending=False).head(10)
print(rising_stars)

# Trouble spots (low rating)
print(f"\n⚠️  TROUBLE SPOTS (Low Rating, Sufficient Reviews)")
trouble = df.groupby(destination_col).agg({
    'Rating': ['mean', 'count'],
    'ensemble_sentiment': lambda x: f"{(x == 'POSITIVE').sum() / len(x) * 100:.1f}%"
}).round(2)
trouble.columns = ['Avg_Rating', 'Review_Count', 'Positive%']
trouble = trouble[trouble['Review_Count'] >= 10]  # At least 10 reviews for credibility
trouble = trouble.sort_values('Avg_Rating', ascending=True).head(10)
print(trouble)

# Quality-Popularity gap
print(f"\n🎯 QUALITY vs POPULARITY GAP")
gap_analysis = df[['destination_rank_by_rating', 'destination_rank_by_reviews', 'popularity_vs_quality_gap']].groupby(df[destination_col]).first()
gap_analysis = gap_analysis.sort_values('popularity_vs_quality_gap', ascending=True).head(10)
gap_analysis.columns = ['Rank_by_Rating', 'Rank_by_Reviews', 'Gap']
print(f"  Overrated (popular but lower quality):")
gap_positive = gap_analysis[gap_analysis['Gap'] > 5]
print(gap_positive.head())

gap_analysis_desc = gap_analysis.sort_values('Gap', ascending=False).head(10)
print(f"\n  Underrated (high quality but less known):")
print(gap_analysis_desc[gap_analysis_desc['Gap'] < -5].head())

print(f"\n📊 FINAL SUMMARY STATISTICS")
print(f"  Total destinations analyzed: {df[destination_col].nunique()}")
print(f"  Avg rating across all: {df['Rating'].mean():.2f}")
print(f"  Top rated destination: {df.groupby(destination_col)['Rating'].mean().idxmax()} ({df.groupby(destination_col)['Rating'].mean().max():.2f})")
print(f"  Most reviewed destination: {df[destination_col].value_counts().index[0]} ({df[destination_col].value_counts().values[0]} reviews)")


## 25. Final Dataset Summary & Export

In [ ]:
print("\n" + "=" * 80)
print("FINAL DATASET SUMMARY & EXPORT")
print("=" * 80)

# Column inventory
print(f"\n📋 FINAL COLUMNS ({len(df.columns)} total)")
print(f"\n  Original columns: 5")
print(f"  Location features: 5")
print(f"  Temporal features: 6")
print(f"  Text measurements: 4")
print(f"  Sentiment/Emotion: 12")
print(f"  Topic modeling: 3")
print(f"  Quality metrics: 6")
print(f"  Destination aggregates: 8")
print(f"  Performance ranking: 3")

# Memory usage
print(f"\n💾 DATASET SIZE")
print(f"  Rows: {len(df):,}")
print(f"  Columns: {len(df.columns)}")
print(f"  Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Data quality
print(f"\n✅ DATA QUALITY METRICS")
print(f"  Complete rows: {len(df) - df.isnull().any(axis=1).sum()} ({(1 - df.isnull().any(axis=1).sum() / len(df)) * 100:.1f}%)")
print(f"  Max missing values in column: {df.isnull().sum().max()}")
print(f"  Duplicates: {df.duplicated().sum()}")

# Key statistics
print(f"\n📊 KEY STATISTICS")
print(f"  Reviews analyzed: {len(df):,}")
print(f"  Rating range: {df['Rating'].min():.0f} - {df['Rating'].max():.0f}")
print(f"  Avg rating: {df['Rating'].mean():.2f} ± {df['Rating'].std():.2f}")
print(f"  Positive reviews: {(df['ensemble_sentiment'] == 'POSITIVE').sum():,} ({(df['ensemble_sentiment'] == 'POSITIVE').mean() * 100:.1f}%)")
print(f"  Inconsistent reviews: {df['inconsistency_flag'].sum():,} ({df['inconsistency_flag'].mean() * 100:.1f}%)")

# Export processed dataset
print(f"\n💾 EXPORTING PROCESSED DATASET...")
output_file = "processed_tourism_reviews_final.csv"
df.to_csv(output_file, index=False, encoding='utf-8')
print(f"  ✅ Saved to: {output_file}")
print(f"  Size: {len(df):,} rows × {len(df.columns)} columns")

# Summary statistics export
print(f"\n📈 EXPORTING SUMMARY STATISTICS...")
summary_stats = {
    'Total_Reviews': len(df),
    'Avg_Rating': df['Rating'].mean(),
    'Positive_Reviews_%': (df['ensemble_sentiment'] == 'POSITIVE').mean() * 100,
    'Negative_Reviews_%': (df['ensemble_sentiment'] == 'NEGATIVE').mean() * 100,
    'Avg_Word_Count': df['word_count'].mean(),
    'Avg_Helpful_Votes': df['Helpful_Votes'].mean(),
    'Unique_Destinations': df[destination_col].nunique() if destination_col in df.columns else 'N/A',
    'Unique_Countries': df['user_country'].nunique(),
    'Date_Range': f"{df['Travel_Date'].min().date()} to {df['Travel_Date'].max().date()}"
}

summary_df = pd.DataFrame(summary_stats, index=[0])
summary_df.to_csv("tourism_analysis_summary.csv", index=False)
print(f"  ✅ Saved to: tourism_analysis_summary.csv")

print(f"\n✨ ANALYSIS COMPLETE!")
print(f"  All 10 analyses executed successfully")
print(f"  Final dataset ready for visualization and reporting")


## 25b. Performance Metrics & Optimization Summary

In [ ]:
print("\n" + "=" * 80)
print("PERFORMANCE METRICS & OPTIMIZATION SUMMARY")
print("=" * 80)

print(f"\n⚡ OPTIMIZATION TECHNIQUES APPLIED")
print(f"  ✓ Batch processing (32 samples/batch) - sentiment & emotion")
print(f"  ✓ GPU acceleration available: {device_name}")
print(f"  ✓ Progress bars with tqdm")
if SAMPLE_SIZE:
    print(f"  ✓ Sampling enabled: {SAMPLE_SIZE} samples")
else:
    print(f"  ✓ Full dataset processing: {len(df):,} reviews")

print(f"\n📊 PERFORMANCE IMPROVEMENTS")
print(f"  • Sentiment analysis: {(len(df) / BATCH_SIZE) / len(df) * 100:.1f}% reduction in API calls")
print(f"  • Emotion detection: {(len(df) / BATCH_SIZE) / len(df) * 100:.1f}% reduction in API calls")
print(f"  • Expected speedup: 10-50x faster than row-by-row processing")
print(f"  • Estimated time (100K reviews): 10-20 minutes vs 80+ minutes")

print(f"\n🎯 FEATURE ENGINEERING COMPLETE")
print(f"  ✓ Text measurements: review_length, word_count, title_length, combined_text")
print(f"  ✓ Sentiment features: 4-model ensemble + emotion detection")
print(f"  ✓ Topic modeling: dominant_topic, topic_probability, topic_keywords")
print(f"  ✓ Keyword extraction: extracted_keywords (aspects)")
print(f"  ✓ Quality metrics: rating_class, length_bucket, sentiment_rating_gap, etc.")
print(f"  ✓ Destination aggregates: rankings, sentiment means")
print(f"  ✓ Total new columns created: {len([c for c in df.columns if c not in ['Rating', 'Title', 'Text', 'Located_City', 'User_Location']])}")

print(f"\n✨ READY FOR ANALYSIS")
print(f"  Dataset shape: {df.shape}")
print(f"  Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"  All 10 analyses can now run at optimized speed")


## 11. Save Enriched Dataset

In [ ]:
# Clean up temporary columns
if 'combined_text' in df.columns:
    df = df.drop(columns=['combined_text'])

# Save final dataset
output_file = "processed_tourism_reviews_with_sentiment.csv"
df.to_csv(output_file, index=False)

print(f"✅ Dataset saved: {output_file}")
print(f"\n📊 Final Statistics:")
print(f"  Rows: {len(df):,}")
print(f"  Columns: {len(df.columns)}")
print(f"\n📋 Feature Categories:")
print(f"  Original: Title, Text, Rating, etc.")
print(f"  Location: province, district, user_country, user_region")
print(f"  Temporal: Travel_Date/Year/Month, Published_Date/Year/Month")
print(f"  Text Features: review_length, word_count, title_length")
print(f"  Sentiment: title_cardiff, text_cardiff, combined_cardiff, combined_siebert, combined_bertweet, absa_best")
print(f"  Emotion: emotion")
print(f"  Ensemble: ensemble_sentiment, ensemble_score")